# Imports

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from Bio import SeqIO

from tqdm import tqdm
import numpy as np

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

from utils.aligner import max_similarity_to_ref
from utils.modlamp_utils import (
    compute_modlamp_metrics,
    p_to_star,
    _aliphatic_index,
    _compute_boman_modlamp,
)

# Variables

In [2]:
FASTAS_DIR = "../data/fastas/"

REFERENCE_CSVS = {
    "dbamp": os.path.join(FASTAS_DIR, "dbAMP3.fasta"),
}

QUERY_FASTAS = {
    "hallucination_boltzgen": os.path.join(FASTAS_DIR, "boltzgen_X_hallucination.fasta"),
    "KPC3_binders": os.path.join(FASTAS_DIR, "boltzgen_KPC3_binders.fasta"),
    "NDM5_binders": os.path.join(FASTAS_DIR, "boltzgen_NDM5_binders.fasta"),
    "dplm1": os.path.join(FASTAS_DIR, "dplm1.fasta"),
    "dplm2": os.path.join(FASTAS_DIR, "dplm2.fasta"),
    "random": os.path.join(FASTAS_DIR, "random_peptides.fasta"),
}

OUTDIR = "../results"
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(OUTDIR + "/figures", exist_ok=True)
os.makedirs(OUTDIR + "/tables", exist_ok=True)


# Load Seuqences

In [3]:
def read_query(path: str) -> pd.DataFrame:
    rows = []
    for rec in SeqIO.parse(path, "fasta"):
        seq = str(rec.seq).strip().upper()
        seq = "".join(c for c in seq if c.isalpha())
        rows.append({"id": rec.id, "seq": seq, "length": len(seq)})
    if not rows:
        raise ValueError(f"No FASTA records found in: {path}")
    return pd.DataFrame(rows)

all_ref = {}
for ref_name, path in REFERENCE_CSVS.items():
    all_ref[ref_name] = read_query(path)['seq'].tolist()
    print(f"Ref '{ref_name}': {len(all_ref[ref_name])} unique sequences")

all_qry = {}
for query_name, qpath in QUERY_FASTAS.items():
    all_qry[query_name] = read_query(qpath)
    print(f"Query '{query_name}': {len(all_qry[query_name])} peptides")

Ref 'dbamp': 35599 unique sequences
Query 'hallucination_boltzgen': 100 peptides
Query 'KPC3_binders': 100 peptides
Query 'NDM5_binders': 100 peptides
Query 'dplm1': 200 peptides
Query 'dplm2': 50 peptides
Query 'random': 280 peptides


In [13]:
for ref_name, ref_seqs in all_ref.items():
    sim_by_query = {}
    for query_name, qry in all_qry.items():
        results = []
        for _, row in tqdm(qry.iterrows(), total=len(qry), desc=f"Similarity: {query_name} vs {ref_name}"):
            qid, qseq = row["id"], row["seq"]
            best_raw, best_idx, best_norm, best_align_norm = max_similarity_to_ref(qseq, ref_seqs)
            results.append(
                {
                    "query_id": qid,
                    "query_len": int(len(qseq)),
                    "max_raw_score": float(best_raw),
                    "max_norm_score_max_len_frac": float(best_norm),
                    "max_norm_score_align_len_frac": float(best_align_norm),
                    "max_similarity_percent": float(best_norm * 100.0),
                }
            )
        sim_df = pd.DataFrame(results)
        sim_by_query[query_name] = sim_df
        safe_q = query_name.replace("/", "_").replace(".", "_")
        safe_r = ref_name.replace("/", "_").replace(".", "_")
        sim_csv = os.path.join(OUTDIR, f"tables/similarity_{safe_q}_{safe_r}.csv")
        sim_df.to_csv(sim_csv, index=False)
        print(f"Saved {sim_csv}")

    safe_r = ref_name.replace("/", "_").replace(".", "_")

    # Combined boxplot: all queries side by side
    fig, ax = plt.subplots(figsize=(max(6, len(sim_by_query) * 1.2), 5))
    data = [sim_by_query[q]["max_similarity_percent"].values for q in sim_by_query]
    labels = list(sim_by_query.keys())
    bp = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)
    for patch in bp["boxes"]:
        patch.set_facecolor("#7eb8da")
    ax.set_ylabel("Max NW similarity to reference (%)")
    ax.set_title(f"Similarity vs {ref_name} (globalxx)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    box_path = os.path.join(OUTDIR, f"figures/similarity_box_{safe_r}.png")
    fig.savefig(box_path, dpi=200)
    plt.close()
    print(f"Saved {box_path}")

    # Combined histogram: all query distributions overlaid
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(sim_by_query)))
    for i, (query_name, sim_df) in enumerate(sim_by_query.items()):
        ax.hist(
            sim_df["max_similarity_percent"],
            bins=40,
            alpha=0.5,
            label=query_name,
            color=colors[i],
            density=True,
        )
    ax.set_xlabel("Max NW similarity to reference peptides (%)")
    ax.set_ylabel("Density")
    ax.set_title(f"Similarity distribution vs {ref_name} (globalxx)")
    ax.legend()
    plt.tight_layout()
    hist_path = os.path.join(OUTDIR, f"figures/similarity_hist_{safe_r}.png")
    fig.savefig(hist_path, dpi=200)
    plt.close()
    print(f"Saved {hist_path}")

Similarity: hallucination_boltzgen vs dbamp: 100%|██████████| 100/100 [00:08<00:00, 12.35it/s]


Saved ../results/tables/similarity_hallucination_boltzgen_dbamp.csv


Similarity: KPC3_binders vs dbamp: 100%|██████████| 100/100 [00:07<00:00, 14.03it/s]


Saved ../results/tables/similarity_KPC3_binders_dbamp.csv


Similarity: NDM5_binders vs dbamp: 100%|██████████| 100/100 [00:07<00:00, 12.60it/s]


...

Saved ../results/tables/similarity_dplm1_dbamp.csv


Similarity: dplm2 vs dbamp: 100%|██████████| 50/50 [00:03<00:00, 13.63it/s]


Saved ../results/tables/similarity_dplm2_dbamp.csv


Similarity: random vs dbamp: 100%|██████████| 280/280 [00:20<00:00, 13.53it/s]
/var/folders/3g/4w5q88kn11l1gcfc1djvcs_w0000gn/T/ipykernel_79145/248748015.py:32: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)


Saved ../results/tables/similarity_random_dbamp.csv
Saved ../results/figures/similarity_box_dbamp.png
Saved ../results/figures/similarity_hist_dbamp.png


In [4]:
for ref_name, ref_seqs in all_ref.items():
    for query_name, qry in all_qry.items():
        safe_q = query_name.replace("/", "_").replace(".", "_")
        safe_r = ref_name.replace("/", "_").replace(".", "_")

        # 1) Compute properties
        train_props = compute_modlamp_metrics(ref_seqs)
        train_props["source"] = "training"
        train_props["seq_id"] = list(range(len(ref_seqs)))

        gen_props = compute_modlamp_metrics(qry["seq"].tolist())
        gen_props["source"] = "generated"
        gen_props["seq_id"] = qry["id"].values

        # 2) Combine + save per-peptide metrics
        props_df = pd.concat([train_props, gen_props], ignore_index=True)
        props_path = os.path.join(OUTDIR, f"modlamp_metrics_{safe_q}_{safe_r}.csv")
        props_df.to_csv(props_path, index=False)

        # 3) Welch's t-test + BH correction
        metrics = [c for c in train_props.columns if c not in ("source", "seq_id")]
        tests = []
        pvals = []
        for metric in metrics:
            train_vals = train_props[metric].astype(float).values
            gen_vals = gen_props[metric].astype(float).values
            if len(gen_vals) < 2 or len(train_vals) < 2:
                tstat, pval = np.nan, np.nan
            else:
                tstat, pval = ttest_ind(gen_vals, train_vals, equal_var=False, nan_policy="omit")
            rec = {
                "metric": metric,
                "training_mean": float(np.nanmean(train_vals)),
                "training_std": float(np.nanstd(train_vals, ddof=1)),
                "generated_mean": float(np.nanmean(gen_vals)),
                "generated_std": float(np.nanstd(gen_vals, ddof=1)),
                "t_stat": float(tstat) if np.isfinite(tstat) else np.nan,
                "p_value": float(pval) if np.isfinite(pval) else np.nan,
            }
            tests.append(rec)
            pvals.append(pval if np.isfinite(pval) else 1.0)
        adj_pvals = multipletests(pvals, method="fdr_bh")[1]
        for rec, adj_p in zip(tests, adj_pvals):
            rec["adjusted_p"] = float(adj_p)
            rec["significance"] = p_to_star(adj_p)
        stats_df = pd.DataFrame(tests)
        stats_path = os.path.join(OUTDIR, f"modlamp_stats_{safe_q}_{safe_r}.csv")
        stats_df.to_csv(stats_path, index=False)
        print(f"Saved {props_path}, {stats_path}")

        # Violin plot
        props_plot = props_df.copy()
        required_cols = ["source", "seq_id", "charge", "isoelectric_point", "instability", "aromaticity", "hydrophobic_ratio"]
        if all(c in props_plot.columns for c in required_cols):
            props_plot["source"] = props_plot["source"].astype(str).str.lower()
            ref_plot = pd.DataFrame({"seq_id": list(range(len(ref_seqs))), "sequence": ref_seqs, "length": len(ref_seqs)})
            ref_plot["source"] = "training"
            qry_plot = qry.rename(columns={"id": "seq_id", "seq": "sequence"})[["seq_id", "sequence", "length"]].copy()
            qry_plot["source"] = "generated"
            lookup = pd.concat([ref_plot, qry_plot], ignore_index=True).drop_duplicates(subset=["source", "seq_id"], keep="first")
            for col in ["length", "sequence"]:
                if col not in props_plot.columns:
                    props_plot[col] = np.nan
            props_plot = props_plot.merge(lookup[["source", "seq_id", "length", "sequence"]], on=["source", "seq_id"], how="left", suffixes=("", "_lk"))
            for col in ["length", "sequence"]:
                lk = f"{col}_lk"
                if lk in props_plot.columns:
                    props_plot[col] = props_plot[col].where(props_plot[col].notna(), props_plot[lk])
                    props_plot.drop(columns=[lk], inplace=True)
            props_plot["length"] = pd.to_numeric(props_plot["length"], errors="coerce")
            props_plot["charge"] = pd.to_numeric(props_plot["charge"], errors="coerce")
            props_plot["charge_density"] = np.where(props_plot["length"] > 0, props_plot["charge"] / props_plot["length"], np.nan)
            props_plot["aliphatic_index"] = props_plot["sequence"].map(_aliphatic_index)
            need_boman = ("boman_index" not in props_plot.columns) or props_plot["boman_index"].isna().any()
            if need_boman:
                seq_mask = props_plot["sequence"].notna() & props_plot["sequence"].astype(str).str.len().gt(0)
                boman_vals = _compute_boman_modlamp(props_plot.loc[seq_mask, "sequence"].astype(str).tolist())
                props_plot.loc[seq_mask, "boman_index"] = boman_vals
            plot_specs = [
                ("charge", "Charge", "(d)"), ("charge_density", "Charge density", "(e)"),
                ("isoelectric_point", "Isoelectric point", "(f)"), ("instability", "Instability index", "(g)"),
                ("aromaticity", "Aromaticity", "(h)"), ("aliphatic_index", "Aliphatic index", "(i)"),
                ("boman_index", "Boman index", "(j)"), ("hydrophobic_ratio", "Hydrophobicity ratio", "(k)"),
            ]
            for key in [k for k, _, _ in plot_specs]:
                props_plot[key] = pd.to_numeric(props_plot[key], errors="coerce")
            raw_p = {}
            for key, _, _ in plot_specs:
                train_vals = props_plot.loc[props_plot["source"] == "training", key].replace([np.inf, -np.inf], np.nan).dropna().values
                gen_vals = props_plot.loc[props_plot["source"] == "generated", key].replace([np.inf, -np.inf], np.nan).dropna().values
                raw_p[key] = ttest_ind(gen_vals, train_vals, equal_var=False, nan_policy="omit")[1] if len(train_vals) >= 2 and len(gen_vals) >= 2 else np.nan
            valid = [k for k, v in raw_p.items() if np.isfinite(v)]
            adj_p = {k: np.nan for k in raw_p}
            if valid:
                for k, ap in zip(valid, multipletests([raw_p[k] for k in valid], method="fdr_bh")[1]):
                    adj_p[k] = float(ap)
            fig, axes = plt.subplots(2, 4, figsize=(18, 8), constrained_layout=True)
            axes = axes.ravel()
            colors = ["#4C72B0", "#DD8452"]
            group_order, labels = ["training", "generated"], ["Training", "Generated"]
            for ax, (key, title, panel) in zip(axes, plot_specs):
                grouped = [props_plot.loc[props_plot["source"] == g, key].replace([np.inf, -np.inf], np.nan).dropna().values for g in group_order]
                if all(len(v) > 0 for v in grouped):
                    vp = ax.violinplot(grouped, positions=[1, 2], widths=0.8, showmeans=False, showmedians=True, showextrema=False)
                    for body, c in zip(vp["bodies"], colors):
                        body.set_facecolor(c); body.set_edgecolor("black"); body.set_alpha(0.7)
                    ax.scatter([1, 2], [float(np.nanmedian(v)) for v in grouped], color="black", s=12, zorder=3)
                    finite = np.concatenate(grouped)
                    y_min, y_max = float(np.nanmin(finite)), float(np.nanmax(finite))
                    pad = (y_max - y_min) * 0.18 if y_max > y_min else 0.1
                    bracket_y = y_max + 0.35 * pad
                    ax.plot([1, 1, 2, 2], [bracket_y, bracket_y + 0.08 * pad, bracket_y + 0.08 * pad, bracket_y], lw=1, c="black")
                    ax.text(1.5, y_max + 0.55 * pad, p_to_star(adj_p.get(key, np.nan)), ha="center", va="bottom", fontsize=10)
                    ax.set_ylim(y_min - 0.1 * pad, y_max + pad)
                else:
                    ax.text(0.5, 0.5, "insufficient data", transform=ax.transAxes, ha="center", va="center", fontsize=9)
                ax.set_xticks([1, 2]); ax.set_xticklabels(labels); ax.set_title(title, fontsize=11); ax.set_ylabel(title)
                ax.text(0.02, 0.98, panel, transform=ax.transAxes, ha="left", va="top", fontweight="bold")
            fig.suptitle(f"{query_name} vs {ref_name}", fontsize=12)
            violin_path = os.path.join(OUTDIR, f"attributes_violin_{safe_q}_{safe_r}.png")
            fig.savefig(violin_path, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved {violin_path}")

Saved ../results/modlamp_metrics_hallucination_boltzgen_dbamp.csv, ../results/modlamp_stats_hallucination_boltzgen_dbamp.csv
Saved ../results/attributes_violin_hallucination_boltzgen_dbamp.png
Saved ../results/modlamp_metrics_KPC3_binders_dbamp.csv, ../results/modlamp_stats_KPC3_binders_dbamp.csv
Saved ../results/attributes_violin_KPC3_binders_dbamp.png
Saved ../results/modlamp_metrics_NDM5_binders_dbamp.csv, ../results/modlamp_stats_NDM5_binders_dbamp.csv
Saved ../results/attributes_violin_NDM5_binders_dbamp.png
Saved ../results/modlamp_metrics_dplm1_dbamp.csv, ../results/modlamp_stats_dplm1_dbamp.csv
Saved ../results/attributes_violin_dplm1_dbamp.png
Saved ../results/modlamp_metrics_dplm2_dbamp.csv, ../results/modlamp_stats_dplm2_dbamp.csv
Saved ../results/attributes_violin_dplm2_dbamp.png
Saved ../results/modlamp_metrics_random_dbamp.csv, ../results/modlamp_stats_random_dbamp.csv
Saved ../results/attributes_violin_random_dbamp.png


## 5.4 Amino Acid Frequency

Normalized frequency of each amino acid in generated vs training peptides.

In [5]:
# Standard 20 amino acids (one-letter)
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

def normalized_aa_freq(seqs):
    """Count each AA across all sequences and normalize so frequencies sum to 1."""
    from collections import Counter
    counter = Counter()
    for s in seqs:
        counter.update(c.upper() for c in s if c.isalpha())
    total = sum(counter.values()) or 1
    return {aa: counter.get(aa, 0) / total for aa in AA_ORDER}

for ref_name, ref_seqs in all_ref.items():
    for query_name, qry in all_qry.items():
        safe_q = query_name.replace("/", "_").replace(".", "_")
        safe_r = ref_name.replace("/", "_").replace(".", "_")
        train_freq = normalized_aa_freq(ref_seqs)
        gen_freq = normalized_aa_freq(qry["seq"].tolist())
        x = np.arange(len(AA_ORDER))
        w = 0.38
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(x - w/2, [gen_freq[a] for a in AA_ORDER], width=w, label="Generated", color="#7eb8da", edgecolor="none")
        ax.bar(x + w/2, [train_freq[a] for a in AA_ORDER], width=w, label="Train", color="#86c67c", edgecolor="none")
        ax.set_xticks(x)
        ax.set_xticklabels(AA_ORDER)
        ax.set_xlabel("Amino Acids")
        ax.set_ylabel("Normalized Frequency")
        ax.set_title(f"{query_name} vs {ref_name}: Amino Acid Frequency")
        ax.legend(loc="upper right")
        ax.set_ylim(0, None)
        ax.yaxis.set_major_locator(plt.MaxNLocator(6))
        ax.grid(axis="y", alpha=0.5)
        plt.tight_layout()
        aa_freq_path = os.path.join(OUTDIR, f"amino_acid_frequency_{safe_q}_{safe_r}.png")
        fig.savefig(aa_freq_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved {aa_freq_path}")

Saved ../results/amino_acid_frequency_hallucination_boltzgen_dbamp.png
Saved ../results/amino_acid_frequency_KPC3_binders_dbamp.png
Saved ../results/amino_acid_frequency_NDM5_binders_dbamp.png
Saved ../results/amino_acid_frequency_dplm1_dbamp.png
Saved ../results/amino_acid_frequency_dplm2_dbamp.png
Saved ../results/amino_acid_frequency_random_dbamp.png
